In [ ]:
# ================================
# STEP 1: Install Gradio
# ================================
!pip install -q gradio

# ================================
# STEP 2: Dummy Database (Tool Layer)
# ================================
students = {
    "101": {"name": "Rahul", "attendance": 85, "marks": 78},
    "102": {"name": "Priya", "attendance": 92, "marks": 88}
}

# ================================
# STEP 3: Tool Functions
# ================================
def get_attendance(student_id):
    if student_id in students:
        return f"📊 Attendance: {students[student_id]['attendance']}%"
    return "❌ Student not found"

def get_marks(student_id):
    if student_id in students:
        return f"📝 Marks: {students[student_id]['marks']}"
    return "❌ Student not found"

# ================================
# STEP 4: Security Layer
# ================================
def secure_access(user_id, requested_id):
    return user_id == requested_id

# ================================
# STEP 5: MCP Agent Logic (UPDATED)
# ================================
def mcp_agent(message, student_id, history):

    if not student_id:
        history.append({"role": "assistant", "content": " Please enter Student ID"})
        return "", history

    # Simulate logged-in user
    user_id = student_id

    # Security Check
    if not secure_access(user_id, student_id):
        history.append({"role": "assistant", "content": " Access Denied"})
        return "", history

    message_lower = message.lower()

    # Tool Invocation
    if "attendance" in message_lower:
        response = get_attendance(student_id)

    elif "marks" in message_lower:
        response = get_marks(student_id)

    elif "hello" in message_lower or "hi" in message_lower:
        response = " Hello! Ask me about attendance or marks."

    else:
        response = " I can help with attendance or marks."

    # Chat history (NEW FORMAT)
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})

    return "", history

# ================================
# STEP 6: Gradio UI (FIXED)
# ================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("#  Student MCP Agent (Secure + Smart)")

    student_id = gr.Textbox(label="Enter Student ID (e.g., 101)")

    chatbot = gr.Chatbot(type="messages", allow_tags=False)

    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        mcp_agent,
        inputs=[msg, student_id, state],
        outputs=[msg, chatbot]
    )

# ================================
# STEP 7: Launch
# ================================
demo.launch(share=True, debug=True)

/tmp/ipykernel_24150/116615371.py:82: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(type="messages", allow_tags=False)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bac27e8d4c2a93aaab.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install -q gradio groq


In [ ]:
import gradio as gr
from groq import Groq

# 🔐 Add your Groq API key here
client = Groq(api_key="YOUR_GROQ_API_KEY")

# ================================
# Dummy Database
# ================================
students = {
    "101": {"name": "Rahul", "attendance": 85, "marks": 78},
    "102": {"name": "Priya", "attendance": 92, "marks": 88}
}

# ================================
# Tools
# ================================
def get_attendance(student_id):
    return f"Attendance: {students[student_id]['attendance']}%"

def get_marks(student_id):
    return f"Marks: {students[student_id]['marks']}"

# ================================
# LLM + MCP Agent
# ================================
def mcp_agent(message, student_id, history):

    if student_id not in students:
        history.append({"role": "assistant", "content": "Invalid Student ID"})
        return "", history

    # Tool calling logic (LLM decides)
    prompt = f"""
    You are a smart student assistant.

    User question: {message}

    If user asks about attendance → reply with: TOOL:attendance
    If user asks about marks → reply with: TOOL:marks
    Otherwise → give normal helpful response.
    """

    llm_response = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[{"role": "user", "content": prompt}]
    )

    decision = llm_response.choices[0].message.content

    # Tool Execution
    if "TOOL:attendance" in decision:
        response = get_attendance(student_id)

    elif "TOOL:marks" in decision:
        response = get_marks(student_id)

    else:
        response = decision

    # History update
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})

    return "", history

# ================================
# UI
# ================================
with gr.Blocks() as demo:

    gr.Markdown("# 🎓 AI Student Assistant (LLM Powered)")

    student_id = gr.Textbox(label="Student ID")

    chatbot = gr.Chatbot(type="messages")

    msg = gr.Textbox()

    state = gr.State([])

    msg.submit(mcp_agent,
               [msg, student_id, state],
               [msg, chatbot])

demo.launch(debug=True)

In [ ]:
# SCENARIO: “Hospital Smart Assistant System”

# 🏥 Background Story
# A large hospital deploys an AI-powered patient assistant.

# 👉 Patients can ask:
# - “What is my appointment schedule?”
# - “What are my latest test results?”

# 👉 Instead of calling reception or logging into multiple portals,
# 👉 AI fetches it instantly, providing secure, real-time updates.

# ================================
# STEP 1: Install Gradio
# ================================
!pip install -q gradio

# ================================
# STEP 2: Dummy Database (Hospital Data)
# ================================
patients = {
    "P101": {
        "name": "Amit",
        "appointment": "Dr. Sharma - 2 April, 10:30 AM",
        "reports": "Blood Test: Normal | Sugar: 95 mg/dL"
    },
    "P102": {
        "name": "Neha",
        "appointment": "Dr. Mehta - 3 April, 1:00 PM",
        "reports": "X-Ray: Clear | Vitamin D: Low"
    }
}

# ================================
# STEP 3: Tool Functions
# ================================
def get_appointment(patient_id):
    if patient_id in patients:
        return f" Appointment: {patients[patient_id]['appointment']}"
    return " Patient not found"

def get_reports(patient_id):
    if patient_id in patients:
        return f" Test Results: {patients[patient_id]['reports']}"
    return " Patient not found"

# ================================
# STEP 4: Security Layer
# ================================
def secure_access(user_id, requested_id):
    return user_id == requested_id

# ================================
# STEP 5: MCP Agent Logic
# ================================
def hospital_agent(message, patient_id, history):

    if not patient_id:
        history.append({"role": "assistant", "content": " Please enter Patient ID"})
        return "", history

    # Simulated login
    user_id = patient_id

    # Security Check
    if not secure_access(user_id, patient_id):
        history.append({"role": "assistant", "content": " Access Denied"})
        return "", history

    message_lower = message.lower()

    # Tool Invocation
    if "appointment" in message_lower:
        response = get_appointment(patient_id)

    elif "test" in message_lower or "report" in message_lower:
        response = get_reports(patient_id)

    elif "hello" in message_lower or "hi" in message_lower:
        response = " Hello! You can ask about appointments or test results."

    else:
        response = " I can help with appointments and test results."

    # Update chat history (modern format)
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})

    return "", history

# ================================
# STEP 6: Gradio UI
# ================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("#  Hospital Smart Assistant (Secure AI Agent)")

    patient_id = gr.Textbox(label="Enter Patient ID (e.g., P101)")

    chatbot = gr.Chatbot(type="messages", allow_tags=False)

    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        hospital_agent,
        inputs=[msg, patient_id, state],
        outputs=[msg, chatbot]
    )

# ================================
# STEP 7: Launch App
# ================================
demo.launch(share=True, debug=True)

/tmp/ipykernel_24150/706847744.py:102: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(type="messages", allow_tags=False)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://190abbf401901125ec.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://190abbf401901125ec.gradio.live


In [ ]:
!pip install -q gradio groq

In [ ]:
import gradio as gr
from groq import Groq

# 🔐 Add your Groq API key
client = Groq(api_key="YOUR_GROQ_API_KEY")

# ================================
# Dummy Database
# ================================
patients = {
    "P101": {
        "name": "Amit",
        "appointment": "Dr. Sharma - 2 April, 10:30 AM",
        "reports": "Blood Test: Normal | Sugar: 95 mg/dL"
    },
    "P102": {
        "name": "Neha",
        "appointment": "Dr. Mehta - 3 April, 1:00 PM",
        "reports": "X-Ray: Clear | Vitamin D: Low"
    }
}

# ================================
# Tool Functions
# ================================
def get_appointment(patient_id):
    return f"📅 Appointment: {patients[patient_id]['appointment']}"

def get_reports(patient_id):
    return f"🧪 Test Results: {patients[patient_id]['reports']}"

# ================================
# Security
# ================================
def secure_access(user_id, requested_id):
    return user_id == requested_id

# ================================
# LLM + MCP Agent
# ================================
def hospital_agent(message, patient_id, history):

    if not patient_id:
        history.append({"role": "assistant", "content": "⚠️ Please enter Patient ID"})
        return "", history

    if patient_id not in patients:
        history.append({"role": "assistant", "content": "❌ Patient not found"})
        return "", history

    user_id = patient_id

    if not secure_access(user_id, patient_id):
        history.append({"role": "assistant", "content": "🚫 Access Denied"})
        return "", history

    # 🔥 LLM Decision Prompt
    prompt = f"""
    You are a hospital assistant AI.

    User query: {message}

    If user asks about appointment → reply EXACTLY: TOOL:appointment
    If user asks about reports/test → reply EXACTLY: TOOL:reports
    Otherwise → give a helpful response.
    """

    llm_response = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[{"role": "user", "content": prompt}]
    )

    decision = llm_response.choices[0].message.content.strip()

    # ================================
    # Tool Execution
    # ================================
    if "TOOL:appointment" in decision:
        response = get_appointment(patient_id)

    elif "TOOL:reports" in decision:
        response = get_reports(patient_id)

    else:
        response = decision

    # Update history
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})

    return "", history

# ================================
# UI
# ================================
with gr.Blocks() as demo:

    gr.Markdown("# 🏥 AI Hospital Assistant (LLM Powered)")

    patient_id = gr.Textbox(label="Enter Patient ID (e.g., P101)")

    chatbot = gr.Chatbot(type="messages")

    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        hospital_agent,
        inputs=[msg, patient_id, state],
        outputs=[msg, chatbot]
    )

# ================================
# Launch
# ================================
demo.launch(debug=True)

In [7]:
# SCENARIO: “Banking Smart Assistant System”

# 🏦 Background Story
# A major bank launches an AI-powered customer assistant.

# 👉 Customers can ask:
# - “What is my account balance?”
# - “Show me my last 5 transactions.”
# - “When is my loan EMI due?”

# 👉 Instead of logging into apps or waiting on customer service calls,
# 👉 AI fetches the information instantly, securely, and in real time.

# ⚙️ Core Idea:
# Just like the hospital and college scenarios, the assistant removes manual checking, centralizes financial data, and makes access instant.

# 💡 Impact:
# - Saves customers time.
# - Reduces load on call centers.
# - Provides personalized financial insights on demand.

import gradio as gr

customers = {
    "C101": {
        "balance": 45230,
        "transactions": ["2000 Grocery", "5000 Rent"],
        "emi": "5 April - 10000"
    }
}

def banking_agent(message, customer_id, history):

    if not customer_id:
        history.append((" Enter Customer ID", ""))
        return "", history

    if customer_id not in customers:
        history.append((" Invalid ID", ""))
        return "", history

    msg = message.lower()

    if "balance" in msg:
        response = f"₹{customers[customer_id]['balance']}"

    elif "transaction" in msg:
        response = "\n".join(customers[customer_id]['transactions'])

    elif "emi" in msg:
        response = customers[customer_id]['emi']

    else:
        response = "Ask balance / transactions / EMI"

    history.append((message, response))
    return "", history


with gr.Blocks() as demo:

    gr.Markdown("# Banking Assistant")

    customer_id = gr.Textbox(label="Customer ID")

    chatbot = gr.Chatbot()

    msg = gr.Textbox()

    state = gr.State([])

    msg.submit(banking_agent,
               [msg, customer_id, state],
               [msg, chatbot])

demo.launch(share=True)


/tmp/ipykernel_24150/1875864001.py:67: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()   # simple version (no error)
/tmp/ipykernel_24150/1875864001.py:67: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot()   # simple version (no error)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://06ffdf84a850acb456.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install -q gradio groq

In [ ]:
import gradio as gr
from groq import Groq

# 🔐 Add your Groq API key
client = Groq(api_key="YOUR_GROQ_API_KEY")

# ================================
# Dummy Database
# ================================
customers = {
    "C101": {
        "balance": 45230,
        "transactions": [
            "₹2000 - Grocery",
            "₹5000 - Rent",
            "₹1200 - Electricity",
            "₹800 - Recharge",
            "₹1500 - Shopping"
        ],
        "emi": "5 April - ₹10,000"
    }
}

# ================================
# Tool Functions
# ================================
def get_balance(customer_id):
    return f"💰 Balance: ₹{customers[customer_id]['balance']}"

def get_transactions(customer_id):
    txns = customers[customer_id]['transactions']
    return "📜 Last Transactions:\n" + "\n".join(txns)

def get_emi(customer_id):
    return f"📅 EMI Due: {customers[customer_id]['emi']}"

# ================================
# LLM + MCP Agent
# ================================
def banking_agent(message, customer_id, history):

    if not customer_id:
        history.append({"role": "assistant", "content": "⚠️ Enter Customer ID"})
        return "", history

    if customer_id not in customers:
        history.append({"role": "assistant", "content": "❌ Invalid Customer ID"})
        return "", history

    # 🔥 LLM decision prompt
    prompt = f"""
    You are a banking assistant AI.

    User query: {message}

    If user asks about balance → reply EXACTLY: TOOL:balance
    If user asks about transactions → reply EXACTLY: TOOL:transactions
    If user asks about EMI or loan → reply EXACTLY: TOOL:emi
    Otherwise → give helpful response.
    """

    llm_response = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[{"role": "user", "content": prompt}]
    )

    decision = llm_response.choices[0].message.content.strip()

    # ================================
    # Tool Execution
    # ================================
    if "TOOL:balance" in decision:
        response = get_balance(customer_id)

    elif "TOOL:transactions" in decision:
        response = get_transactions(customer_id)

    elif "TOOL:emi" in decision:
        response = get_emi(customer_id)

    else:
        response = decision

    # Update history (modern format)
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})

    return "", history

# ================================
# UI
# ================================
with gr.Blocks() as demo:

    gr.Markdown("# 🏦 AI Banking Assistant (LLM Powered)")

    customer_id = gr.Textbox(label="Customer ID (e.g., C101)")

    chatbot = gr.Chatbot(type="messages")

    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        banking_agent,
        inputs=[msg, customer_id, state],
        outputs=[msg, chatbot]
    )

# ================================
# Launch
# ================================
demo.launch(debug=True)

In [ ]:
# ======================================
# STEP 1: Install Libraries
# ======================================
!pip install groq gradio


# ======================================
# STEP 2: Load API Key from Colab Secrets
# ======================================
from google.colab import userdata

groq_api_key = userdata.get("GROQ_API_KEY")

from groq import Groq
client = Groq(api_key=groq_api_key)


# ======================================
# STEP 3: Dummy Database (Tool Layer)
# ======================================
students = {
    "101": {"name": "Rahul", "attendance": 85, "marks": 78},
    "102": {"name": "Priya", "attendance": 92, "marks": 88}
}


# ======================================
# STEP 4: Tool Functions
# ======================================
def get_attendance(student_id):
    if student_id in students:
        return f" Attendance: {students[student_id]['attendance']}%"
    return " Student not found"


def get_marks(student_id):
    if student_id in students:
        return f" Marks: {students[student_id]['marks']}"
    return " Student not found"


# ======================================
# STEP 5: MCP Tool Decision via LLM
# ======================================
def decide_tool(query):
    try:
        prompt = f"""
        You are an AI assistant.

        Decide which function to call:
        - get_attendance
        - get_marks

        Rules:
        - If user asks about attendance → get_attendance
        - If user asks about marks → get_marks

        Only return function name.

        Query: {query}
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        tool = response.choices[0].message.content.strip().lower()

        return tool

    except Exception as e:
        print(" Groq Error:", e)
        return "fallback"


# ======================================
# STEP 6: MCP Agent (CORE LOGIC)
# ======================================
def mcp_agent(message, student_id, history):

    # Validate input
    if not student_id:
        response = " Please enter Student ID"
        history.append((message, response))
        return "", history

    # Step 1: LLM decides tool
    tool = decide_tool(message)

    # Step 2: Tool Invocation
    if "attendance" in tool:
        response = get_attendance(student_id)

    elif "marks" in tool:
        response = get_marks(student_id)

    # Fallback (if LLM fails)
    elif tool == "fallback":
        if "attendance" in message.lower():
            response = get_attendance(student_id)
        elif "marks" in message.lower():
            response = get_marks(student_id)
        else:
            response = " LLM failed, and I couldn't understand."

    else:
        response = " I can help with attendance or marks."

    # Step 3: Save chat
    history.append((message, response))

    return "", history


# ======================================
# STEP 7: Gradio UI
# ======================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("#  MCP Agent with Groq (Stable Version)")

    student_id = gr.Textbox(label="Enter Student ID (101 / 102)")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        mcp_agent,
        inputs=[msg, student_id, state],
        outputs=[msg, chatbot]
    )


# ======================================
# STEP 8: Launch App
# ======================================
demo.launch(share=True)

In [ ]:
# SCENARIO: “AI Banking Assistant with Role-Based Access”
# 🏦 Background Story

# A bank builds an AI assistant to help users:

# Check account balance
# View transactions
# Approve loans
# Manage customer accounts

# 👉 But not everyone can do everything

# 🧑‍🤝‍🧑 Roles in the Bank
# Role	Description
# 👤 Customer	Bank account holder
# 👨‍💼 Employee	Bank staff
# 🧑‍💼 Manager	Branch manager
# 🔐 Permissions (RBAC)
# Action	Customer	Employee	Manager
# View own balance	✅	✅	✅
# View others' accounts	❌	✅	✅
# Approve loan	❌	❌	✅
# View all transactions	❌	✅	✅


# ======================================
# STEP 1: Install Libraries
# ======================================
!pip install groq gradio


# ======================================
# STEP 2: Load API Key from Colab Secrets
# ======================================
from google.colab import userdata

groq_api_key = userdata.get("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError(" GROQ_API_KEY not found in Colab Secrets")

from groq import Groq
client = Groq(api_key=groq_api_key)


# ======================================
# STEP 3: Dummy Bank Database
# ======================================
accounts = {
    "1001": {"name": "Amit", "balance": 50000},
    "1002": {"name": "Neha", "balance": 75000}
}


# ======================================
# STEP 4: Tool Functions (APIs)
# ======================================
def get_balance(account_id):
    if account_id in accounts:
        return f" Balance of {account_id}: ₹{accounts[account_id]['balance']}"
    return " Account not found"


def approve_loan(account_id):
    if account_id in accounts:
        return f" Loan approved for account {account_id}"
    return " Account not found"


# ======================================
# STEP 5: RBAC Security Layer
# ======================================
def secure_access(role, user_account, requested_account, action):

    # Manager → full access
    if role == "manager":
        return True

    # Employee → can view all but cannot approve loans
    elif role == "employee":
        if action == "approve_loan":
            return False
        return True

    # Customer → only own account, no loan approval
    elif role == "customer":
        return user_account == requested_account and action != "approve_loan"

    return False


# ======================================
# STEP 6: MCP Tool Decision via LLM
# ======================================
def decide_action(query):
    try:
        prompt = f"""
        You are an AI banking assistant.

        Decide which action to take:
        - get_balance
        - approve_loan

        Rules:
        - Balance related queries → get_balance
        - Loan approval queries → approve_loan

        Return ONLY the action name.

        Query: {query}
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        action = response.choices[0].message.content.strip().lower()
        return action

    except Exception as e:
        print(" Groq Error:", e)
        return "fallback"


# ======================================
# STEP 7: MCP Agent (Core Logic)
# ======================================
def banking_agent(message, role, user_account, requested_account, history):

    # Input validation
    if not user_account:
        response = " Please enter your account ID"
        history.append((message, response))
        return "", history

    if not requested_account:
        requested_account = user_account  # default to own account

    # Step 1: LLM decides action
    action = decide_action(message)

    # Step 2: RBAC Security Check
    if not secure_access(role, user_account, requested_account, action):
        response = " Access Denied: You are not authorized"

    else:
        # Step 3: Tool Invocation
        if "balance" in action:
            response = get_balance(requested_account)

        elif "loan" in action:
            response = approve_loan(requested_account)

        # Fallback if LLM fails
        elif action == "fallback":
            msg_lower = message.lower()
            if "balance" in msg_lower:
                response = get_balance(requested_account)
            elif "loan" in msg_lower:
                response = approve_loan(requested_account)
            else:
                response = " Could not understand request"

        else:
            response = " Try asking about balance or loan"

    # Save chat history
    history.append((message, response))

    return "", history


# ======================================
# STEP 8: Gradio UI
# ======================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("#  AI Banking Assistant (MCP + RBAC + Groq)")

    role = gr.Dropdown(
        ["customer", "employee", "manager"],
        label="Select Role"
    )

    user_account = gr.Textbox(label="Your Account ID (e.g., 1001)")
    requested_account = gr.Textbox(label="Target Account ID (optional)")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        banking_agent,
        inputs=[msg, role, user_account, requested_account, state],
        outputs=[msg, chatbot]
    )


# ======================================
# STEP 9: Launch App
# ======================================
demo.launch(share=True)

In [ ]:
# SCENARIO: “University Smart Assistant with Role-Based Access”

#  Background Story
# A university deploys an AI-powered academic assistant to help students, faculty, and administrators.

# Users can ask:
# - “What is my attendance record?”
# - “Show me my exam results.”
# - “Update course schedules.”
# - “Approve new course registrations.”

#  But not everyone can do everything — access depends on roles.

!pip install -q gradio groq
import gradio as gr
from groq import Groq

#  Add your Groq API key
client = Groq(api_key="YOUR_GROQ_API_KEY")

# ================================
# Database
# ================================
users = {
    "S101": {"role": "student", "attendance": 88, "marks": 82},
    "F201": {"role": "faculty"},
    "A301": {"role": "admin"}
}

# ================================
# Tools
# ================================
def get_attendance(user_id):
    return f" Attendance: {users[user_id]['attendance']}%"

def get_marks(user_id):
    return f" Marks: {users[user_id]['marks']}"

def update_schedule():
    return " Course schedule updated successfully"

def approve_registration():
    return " Course registration approved"

# ================================
# RBAC (Role-Based Access Control)
# ================================
def check_access(role, action):
    permissions = {
        "student": ["attendance", "marks"],
        "faculty": ["attendance", "marks", "schedule"],
        "admin": ["attendance", "marks", "schedule", "approve"]
    }
    return action in permissions.get(role, [])

# ================================
# MCP + LLM Agent
# ================================
def university_agent(message, user_id, history):

    if not user_id:
        history.append({"role": "assistant", "content": " Enter User ID"})
        return "", history

    if user_id not in users:
        history.append({"role": "assistant", "content": " Invalid User"})
        return "", history

    role = users[user_id]["role"]

    # LLM decides intent
    prompt = f"""
    You are a university assistant AI.

    User role: {role}
    User query: {message}

    If attendance → reply: TOOL:attendance
    If marks/results → reply: TOOL:marks
    If update schedule → reply: TOOL:schedule
    If approve registration → reply: TOOL:approve
    Otherwise → normal response
    """

    llm_response = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[{"role": "user", "content": prompt}]
    )

    decision = llm_response.choices[0].message.content.strip()

    # ================================
    # Tool Execution + RBAC Check
    # ================================
    if "TOOL:attendance" in decision:
        if check_access(role, "attendance"):
            response = get_attendance(user_id)
        else:
            response = "Access Denied"

    elif "TOOL:marks" in decision:
        if check_access(role, "marks"):
            response = get_marks(user_id)
        else:
            response = "Access Denied"

    elif "TOOL:schedule" in decision:
        if check_access(role, "schedule"):
            response = update_schedule()
        else:
            response = "Access Denied"

    elif "TOOL:approve" in decision:
        if check_access(role, "approve"):
            response = approve_registration()
        else:
            response = "Access Denied"

    else:
        response = decision

    # Update history
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})

    return "", history

# ================================
# UI
# ================================
with gr.Blocks() as demo:

    gr.Markdown("# University Smart Assistant (RBAC + AI)")

    user_id = gr.Textbox(label="Enter User ID (S101 / F201 / A301)")

    chatbot = gr.Chatbot(type="messages")

    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        university_agent,
        inputs=[msg, user_id, state],
        outputs=[msg, chatbot]
    )

# ================================
# Launch
# ================================
demo.launch(debug=True)

In [ ]:
# SCENARIO: “Retail Smart Assistant with Role-Based Access”

# 🏬 Background Story
# A large retail chain introduces an AI-powered store assistant to streamline operations.

# 👉 Users can ask:
# - “What is my purchase history?”
# - “Check inventory for product X.”
# - “Approve supplier orders.”
# - “Manage employee schedules.”

# 👉 But not everyone can do everything — access depends on roles.

import gradio as gr
from groq import Groq

# 🔐 Add your Groq API key
client = Groq(api_key="YOUR_GROQ_API_KEY")

# ================================
# Database
# ================================
users = {
    "U101": {"role": "customer", "history": ["Shoes", "T-shirt", "Watch"]},
    "S201": {"role": "staff"},
    "M301": {"role": "manager"}
}

inventory = {
    "shoes": 20,
    "t-shirt": 50,
    "watch": 15
}

# ================================
# Tools
# ================================
def get_purchase_history(user_id):
    return "🛍️ Purchase History:\n" + "\n".join(users[user_id]["history"])

def check_inventory(product):
    product = product.lower()
    if product in inventory:
        return f" {product} stock: {inventory[product]}"
    return " Product not found"

def approve_order():
    return " Supplier order approved"

def manage_schedule():
    return " Employee schedules updated"

# ================================
# RBAC
# ================================
def check_access(role, action):
    permissions = {
        "customer": ["history"],
        "staff": ["history", "inventory"],
        "manager": ["history", "inventory", "approve", "schedule"]
    }
    return action in permissions.get(role, [])

# ================================
# MCP + LLM Agent
# ================================
def retail_agent(message, user_id, history):

    if not user_id:
        history.append({"role": "assistant", "content": " Enter User ID"})
        return "", history

    if user_id not in users:
        history.append({"role": "assistant", "content": " Invalid User"})
        return "", history

    role = users[user_id]["role"]

    #  LLM decides intent
    prompt = f"""
    You are a retail assistant AI.

    User role: {role}
    User query: {message}

    If purchase history → reply: TOOL:history
    If inventory check → reply: TOOL:inventory
    If approve order → reply: TOOL:approve
    If manage schedule → reply: TOOL:schedule
    Also extract product name if mentioned.
    Otherwise → normal response.
    """

    llm_response = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[{"role": "user", "content": prompt}]
    )

    decision = llm_response.choices[0].message.content.strip().lower()

    # ================================
    # Tool Execution + RBAC
    # ================================
    if "tool:history" in decision:
        if check_access(role, "history"):
            response = get_purchase_history(user_id)
        else:
            response = " Access Denied"

    elif "tool:inventory" in decision:
        if check_access(role, "inventory"):
            # simple extraction
            words = message.lower().split()
            product = next((w for w in words if w in inventory), None)

            if product:
                response = check_inventory(product)
            else:
                response = " Please specify product name"
        else:
            response = " Access Denied"

    elif "tool:approve" in decision:
        if check_access(role, "approve"):
            response = approve_order()
        else:
            response = " Access Denied"

    elif "tool:schedule" in decision:
        if check_access(role, "schedule"):
            response = manage_schedule()
        else:
            response = " Access Denied"

    else:
        response = decision

    # Update history
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})

    return "", history

# ================================
# UI
# ================================
with gr.Blocks() as demo:

    gr.Markdown("#  Retail Smart Assistant (RBAC + AI)")

    user_id = gr.Textbox(label="User ID (U101 / S201 / M301)")

    chatbot = gr.Chatbot(type="messages")

    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        retail_agent,
        inputs=[msg, user_id, state],
        outputs=[msg, chatbot]
    )

# ================================
# Launch
# ================================
demo.launch(debug=True)

In [ ]:
# ======================================
# STEP 1: Install Libraries
# ======================================
!pip install groq nest_asyncio


# ======================================K
# STEP 2: Load Groq API Key (Colab Secret)
# ======================================
from google.colab import userdata
groq_api_key = userdata.get("GROQ_API_KEY")

from groq import Groq
client = Groq(api_key=groq_api_key)


# ======================================
# STEP 3: MOCK TOOLS (Simulating MCP Tools)
# ======================================

import asyncio
import random
import nest_asyncio

# Apply nest_asyncio to allow nested event loops
nest_asyncio.apply()

async def web_search(query):
    await asyncio.sleep(1)  # simulate delay
    return f"📰 News about {query}: Market is growing fast."

async def get_stock_data(company):
    await asyncio.sleep(1)
    price = random.randint(100, 500)
    return f"📈 Stock price of {company}: ${price}"

async def fetch_company_profile(company):
    await asyncio.sleep(1)
    return f"👥 {company} has 5000 employees, HQ in USA"


# ======================================
# STEP 4: PARALLEL TOOL INVOCATION
# ======================================

async def parallel_research(company):

    results = await asyncio.gather(
        web_search(company),
        get_stock_data(company),
        fetch_company_profile(company),
        return_exceptions=True
    )

    news, stock, profile = results

    return {
        "news": news if not isinstance(news, Exception) else "News unavailable",
        "stock": stock if not isinstance(stock, Exception) else "Stock unavailable",
        "profile": profile if not isinstance(profile, Exception) else "Profile unavailable"
    }


# ======================================
# STEP 5: CHAINED TOOL INVOCATION USING GROQ
# ======================================

def analyse_text(text):
    response = client.chat.completions.create(
    model="llama-3.3-70b-versatile", # Updated model name to a currently available Groq model
        messages=[{
            "role": "user",
            "content": f"Analyze this data and give key insights:\n{text}"
        }]
    )
    return response.choices[0].message.content


def generate_report(analysis, company):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile", # Updated model name to a currently available Groq model
        messages=[{
            "role": "user",
            "content": f"Create a professional report for {company}:\n{analysis}"
        }]
    )
    return response.choices[0].message.content


# ======================================
# STEP 6: FULL MCP PIPELINE
# ======================================

async def full_pipeline(company):

    # Step 1: Parallel Data Collection
    data = await parallel_research(company)

    combined_text = f"""
    News: {data['news']}
    Stock: {data['stock']}
    Profile: {data['profile']}
    """

    # Step 2: Analysis (LLM)
    analysis = analyse_text(combined_text)

    # Step 3: Report Generation (LLM)
    report = generate_report(analysis, company)

    return report


# ======================================
# STEP 7: RUN THE SYSTEM
# ======================================

company_name = "Tesla"

# Use asyncio.run() after applying nest_asyncio
result = asyncio.run(full_pipeline(company_name))

print("📊 FINAL REPORT:\n")
print(result)

In [ ]:
# SCENARIO: “Corporate Research Assistant System”

# 🏢 Background Story
# A multinational company deploys an AI-powered business intelligence assistant.

# 👉 Employees can ask:
# - “What’s the latest news about Tesla?”
# - “What’s Tesla’s current stock price?”
# - “Give me a company profile instantly.”

# 👉 Instead of manually searching news sites, finance portals, and HR databases,
# 👉 AI fetches all the data in parallel, analyzes it, and generates a professional report.

# ⚙️ How it works (mapped to your pipeline):
# - Parallel Data Collection → AI gathers news, stock prices, and company profiles simultaneously.
# - LLM Analysis → AI interprets the combined data, highlighting key insights.
# - Report Generation → AI produces a polished, executive-ready report.

# 💡 Impact:
# - Saves analysts hours of manual research.
# - Provides real-time, consolidated insights for decision-making.
# - Empowers managers with instant reports for board meetings or investor updates.
!pip install -q gradio groq yfinance
import gradio as gr
from groq import Groq
import yfinance as yf

# 🔐 Add your Groq API key
client = Groq(api_key="YOUR_GROQ_API_KEY")

# ================================
# TOOL 1: Stock Price
# ================================
def get_stock_price(company):
    try:
        stock = yf.Ticker(company)
        price = stock.history(period="1d")["Close"][0]
        return f"📈 Stock Price: ${round(price,2)}"
    except:
        return "Stock data not available"

# ================================
# TOOL 2: News (Dummy)
# ================================
def get_news(company):
    return f"""
📰 Latest News:
- {company} shows strong growth
- Expansion in global markets
- Investors optimistic
"""

# ================================
# TOOL 3: Company Profile
# ================================
def get_profile(company):
    return f"""
🏢 Company: {company}
Industry: Tech / Automotive
Overview: Leading innovative company with global presence
"""

# ================================
# MCP + LLM Agent
# ================================
def research_agent(message, history):

    company = message.strip()

    # ================================
    # PARALLEL DATA COLLECTION
    # ================================
    stock = get_stock_price(company)
    news = get_news(company)
    profile = get_profile(company)

    # ================================
    # LLM REPORT GENERATION
    # ================================
    prompt = f"""
    You are a corporate analyst.

    Company: {company}

    Data:
    {stock}
    {news}
    {profile}

    Create a professional report with:
    - Summary
    - Key Insights
    - Market Outlook
    - Conclusion
    """

    llm = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[{"role": "user", "content": prompt}]
    )

    report = llm.choices[0].message.content

    # Chat history
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": report})

    return "", history

# ================================
# UI
# ================================
with gr.Blocks() as demo:

    gr.Markdown("# 🏢 Corporate Research Assistant")

    chatbot = gr.Chatbot(type="messages")

    msg = gr.Textbox(label="Enter Company Name (e.g., TSLA)")

    state = gr.State([])

    msg.submit(
        research_agent,
        inputs=[msg, state],
        outputs=[msg, chatbot]
    )

# ================================
# Launch
# ================================
demo.launch(debug=True)

In [ ]:
# 🏥 SCENARIO: “Healthcare Research Assistant System”

# 🩺 Background Story
# A medical research institute deploys an AI-powered clinical intelligence assistant.

# 👉 Researchers and doctors can ask:
# • “What’s the latest research on diabetes treatments?”
# • “Summarize recent clinical trial results for cancer drugs.”
# • “Give me a profile of a pharmaceutical company instantly.”

# 👉 Instead of manually searching journals, trial databases, and company reports,
# 👉 AI fetches all the data in parallel, analyzes it, and generates a professional research summary.

# ⚙️ How it works (mapped to your pipeline):
# • Parallel Data Collection → AI gathers medical news, trial results, and company profiles simultaneously.
# • LLM Analysis → AI interprets the combined data, highlighting key medical insights.
# • Report Generation → AI produces a polished, researcher-ready report.

!pip install -q gradio groq
import gradio as gr
from groq import Groq

# 🔐 Add your Groq API key
client = Groq(api_key="YOUR_GROQ_API_KEY")

# ================================
# TOOL 1: Medical News (Dummy)
# ================================
def get_medical_news(topic):
    return f"""
📰 Latest Research News on {topic}:
- New breakthrough therapies under development
- Improved patient outcomes reported
- Increased global research funding
"""

# ================================
# TOOL 2: Clinical Trials (Dummy)
# ================================
def get_clinical_trials(topic):
    return f"""
🧪 Clinical Trials:
- Phase 3 trials show promising results
- Reduced side effects observed
- Ongoing global trials expanding
"""

# ================================
# TOOL 3: Pharma Company Profile
# ================================
def get_pharma_profile(topic):
    return f"""
🏢 Pharma Insights:
- Leading companies investing in {topic}
- Strong R&D pipelines
- Strategic partnerships in progress
"""

# ================================
# MCP + LLM Agent
# ================================
def healthcare_agent(message, history):

    topic = message.strip()

    # ================================
    # PARALLEL DATA COLLECTION
    # ================================
    news = get_medical_news(topic)
    trials = get_clinical_trials(topic)
    pharma = get_pharma_profile(topic)

    # ================================
    # LLM ANALYSIS + REPORT
    # ================================
    prompt = f"""
    You are a medical research assistant.

    Topic: {topic}

    Data:
    {news}
    {trials}
    {pharma}

    Generate a professional medical research summary including:
    - Overview
    - Key Findings
    - Clinical Insights
    - Future Scope
    """

    llm = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[{"role": "user", "content": prompt}]
    )

    report = llm.choices[0].message.content

    # Update history
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": report})

    return "", history

# ================================
# UI
# ================================
with gr.Blocks() as demo:

    gr.Markdown("# 🩺 Healthcare Research Assistant (AI Powered)")

    chatbot = gr.Chatbot(type="messages")

    msg = gr.Textbox(label="Enter Topic (e.g., diabetes, cancer drugs)")

    state = gr.State([])

    msg.submit(
        healthcare_agent,
        inputs=[msg, state],
        outputs=[msg, chatbot]
    )

# ================================
# Launch
# ================================
demo.launch(debug=True)


In [ ]:
# 🖥️ SCENARIO: “AI IT Helpdesk Assistant in a Large Company”
# 🏢 Background Story

# A large company (like Infosys or TCS) has thousands of employees.

# 👉 Employees face issues daily:

# VPN not working
# System hacked
# Email access denied
# Network outage

# 👉 Instead of manual IT support, the company builds an:

# 🤖 AI IT Helpdesk Assistant


# ======================================
# STEP 1: Install Dependencies
# ======================================
!pip install groq


# ======================================
# STEP 2: Load API Key from Colab Secrets
# ======================================
from google.colab import userdata
from groq import Groq
import asyncio
import random
import logging

logging.basicConfig(level=logging.INFO)

groq_api_key = userdata.get("GROQ_API_KEY")
client = Groq(api_key=groq_api_key)


# ======================================
# STEP 3: Simulated MCP Tools (Mock APIs)
# ======================================
async def page_security_team(payload):
    return "🚨 Security team paged"

async def create_jira_ticket(payload):
    return f"🎫 Jira ticket created: {payload}"

async def check_network_status(payload):
    return random.choice(["Network degraded", "All systems normal"])

async def alert_noc_team(payload):
    return "📡 NOC team alerted"

async def get_ad_user(payload):
    return "user_123"

async def reset_permissions(payload):
    return "🔐 Permissions reset successfully"

async def query_postgres(payload):
    if random.random() < 0.7:
        raise Exception("Database timeout")
    return "📦 Data from Postgres"

async def query_sqlite_cache(payload):
    return "🗂 Data from SQLite cache (fallback)"


# ======================================
# STEP 4: LLM Classifier via Groq
# ======================================
def classify_issue(issue):
    prompt = f"""
    Classify the following IT issue into ONE category only:
    network, hardware, software, security, access

    Issue: {issue}

    Return only the category name.
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content.strip().lower()


# ======================================
# STEP 5: Conditional Routing Logic
# ======================================
async def smart_it_triage(issue, severity):
    issue_type = classify_issue(issue)

    print(f"🧠 Classified as: {issue_type}")

    if issue_type == "security":
        await page_security_team({"issue": issue})
        return await create_jira_ticket({"project": "SEC", "priority": "Blocker"})

    elif issue_type == "network" and severity == "high":
        status = await check_network_status({})
        if "degraded" in status.lower():
            return await alert_noc_team({"issue": issue})
        else:
            return await create_jira_ticket({"project": "NET", "priority": "High"})

    elif issue_type == "access":
        user = await get_ad_user({"query": issue})
        return await reset_permissions({"user": user})

    else:
        return await create_jira_ticket({"project": "IT", "priority": "Medium"})


# ======================================
# STEP 6: Retry with Exponential Backoff
# ======================================
async def call_tool_with_retry(
    primary_tool,
    fallback_tool=None,
    max_retries=3
):
    delay = 1

    for attempt in range(max_retries):
        try:
            return await primary_tool({})
        except Exception as e:
            logging.warning(f"Attempt {attempt+1} failed: {e}")

            if attempt == max_retries - 1:
                if fallback_tool:
                    logging.info("Switching to fallback tool")
                    return await fallback_tool({})
                raise

            await asyncio.sleep(delay)
            delay *= 2


# ======================================
# STEP 7: Run Demo
# ======================================
async def run_demo():
    print("=== Smart IT Triage Demo ===")

    result = await smart_it_triage(
        issue="VPN not working for employee",
        severity="high"
    )

    print("🔍 Routing Result:", result)

    print("\n=== Retry Pattern Demo ===")

    data = await call_tool_with_retry(
        query_postgres,
        fallback_tool=query_sqlite_cache
    )

    print("📊 Data Result:", data)


# ======================================
# STEP 8: Execute (Colab-safe)
# ======================================
await run_demo()

In [ ]:
# SCENARIO: “AI Healthcare Support Assistant in a Large Hospital”

# 🏢 Background Story
# A large hospital (like Apollo or Fortis) has thousands of patients and staff members.

# 👉 Daily Challenges Faced:
# - Patients waiting hours for appointment confirmations
# - Confusion about lab test results and reports
# - Doctors overwhelmed with scheduling and follow-up reminders
# - Nurses struggling to track medicine administration times
# - Emergency cases needing instant triage

# 👉 Instead of manual coordination, the hospital builds an:
# 🤖 AI Healthcare Support Assistant

# 🚑 Capabilities:
# - 📅 Smart Scheduling: Automatically books and reschedules patient appointments based on doctor availability.
# - 🧪 Lab Report Explainer: Summarizes test results in simple language for patients.
# - 💊 Medication Tracker: Sends reminders to nurses and patients about dosage timings.
# - 🚨 Emergency Triage: Instantly categorizes incoming cases (critical, urgent, routine) and alerts the right medical team.
# - 📧 Follow-up Automation: Sends personalized recovery instructions and reminders after discharge.

# 🎯 Impact:
# - Reduced patient waiting time
# - Doctors spend more time on treatment, less on admin work
# - Nurses avoid errors in medicine administration
# - Faster response in emergencies
# - Improved patient satisfaction and trust

!pip install -q gradio groq
import gradio as gr
from groq import Groq

# 🔐 Add your Groq API key
client = Groq(api_key="YOUR_GROQ_API_KEY")

# ================================
# Dummy Data
# ================================
appointments = {}
medications = {"P101": "Take Paracetamol at 8 AM & 8 PM"}

# ================================
# TOOLS
# ================================

# 📅 Scheduling
def book_appointment(patient_id):
    appointments[patient_id] = "Booked for Tomorrow 10 AM"
    return f"📅 Appointment booked: {appointments[patient_id]}"

# 🧪 Lab Report Explainer
def explain_report(report):
    return f"🧪 Simplified Report: {report} looks normal. No major issues detected."

# 💊 Medication Reminder
def medication_reminder(patient_id):
    return f"💊 Reminder: {medications.get(patient_id, 'No medication found')}"

# 🚨 Emergency Triage
def triage_case(symptoms):
    if "chest pain" in symptoms or "breathing" in symptoms:
        return "🚨 CRITICAL: Immediate attention required!"
    elif "fever" in symptoms:
        return "⚠️ URGENT: Doctor consultation needed soon."
    else:
        return "🟢 ROUTINE: No immediate risk."

# 📧 Follow-up
def follow_up(patient_id):
    return "📧 Follow-up: Take rest, stay hydrated, revisit after 7 days."

# ================================
# MCP + LLM AGENT
# ================================
def healthcare_agent(message, patient_id, history):

    if not patient_id:
        history.append({"role": "assistant", "content": "⚠️ Enter Patient ID"})
        return "", history

    # 🔥 LLM decides intent
    prompt = f"""
    You are an AI healthcare assistant.

    User message: {message}

    If appointment → reply: TOOL:appointment
    If report → reply: TOOL:report
    If medicine → reply: TOOL:medicine
    If emergency → reply: TOOL:triage
    If follow-up → reply: TOOL:followup
    Otherwise → normal helpful reply
    """

    llm = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[{"role": "user", "content": prompt}]
    )

    decision = llm.choices[0].message.content.lower()

    # ================================
    # TOOL EXECUTION
    # ================================
    if "tool:appointment" in decision:
        response = book_appointment(patient_id)

    elif "tool:report" in decision:
        response = explain_report(message)

    elif "tool:medicine" in decision:
        response = medication_reminder(patient_id)

    elif "tool:triage" in decision:
        response = triage_case(message.lower())

    elif "tool:followup" in decision:
        response = follow_up(patient_id)

    else:
        response = decision

    # Update history
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": response})

    return "", history

# ================================
# UI
# ================================
with gr.Blocks() as demo:

    gr.Markdown("# 🏥 AI Healthcare Support Assistant")

    patient_id = gr.Textbox(label="Patient ID (e.g., P101)")

    chatbot = gr.Chatbot(type="messages")

    msg = gr.Textbox(label="Ask your query")

    state = gr.State([])

    msg.submit(
        healthcare_agent,
        inputs=[msg, patient_id, state],
        outputs=[msg, chatbot]
    )

# ================================
# Launch
# ================================
demo.launch(debug=True)